In [1]:
import json
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from transformers import pipeline

c:\Users\usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:


# ===================================
# Carregar dados e preparar o modelo
# ===================================
dataset_synopsis_treated = pd.read_parquet('../data/dataset_synopsis_treated.parquet')

# Stopwords (palavras para ignorar)
pt_stop_words = ['de', 'a', 'o', 'que', 'e', 'do', 'da', 'em', 'um', 'para', 'com', 'não', 'uma', 'os', 'no', 'se', 'na', 'por', 'mais', 'as', 'dos', 'como', 'mas', 'ao', 'ele', 'das', 'à', 'seu', 'sua', 'ou', 'quando', 'muito', 'nos', 'já', 'eu', 'também', 'só', 'pelo', 'pela', 'até', 'isso', 'ela', 'entre', 'depois', 'sem', 'mesmo', 'aos', 'seus', 'quem', 'nas', 'me', 'esse', 'eles', 'está', 'você', 'tinha', 'foram', 'essa', 'num', 'nem', 'suas', 'meu', 'às', 'minha', 'têm', 'numa', 'pelos', 'elas', 'qual', 'nós', 'lhe', 'deles', 'essas', 'esses', 'pelas', 'este', 'dele', 'tu', 'te', 'vocês', 'vos', 'lhes', 'meus', 'minhas', 'teu', 'tua', 'teus', 'tuas', 'nosso', 'nossa', 'nossos', 'nossas', 'dela', 'delas', 'esta', 'estes', 'estas', 'aquele', 'aquela', 'aqueles', 'aquelas', 'isto', 'aquilo', 'sobre', 'filme', 'história', 'vida', 'personagem', 'conta']

# Criar a matriz TF-IDF (transforma texto em números)
tfidf_vectorizer = TfidfVectorizer(max_features=1500, stop_words=pt_stop_words, ngram_range=(1, 2))
tfidf_matrix = tfidf_vectorizer.fit_transform(dataset_synopsis_treated['synopsis'])


# Criar o modelo para achar 5 temas
nmf_model = NMF(n_components=10, random_state=42)

# Treinar o modelo com dados
nmf_model.fit(tfidf_matrix)


# ================================
# Nomeação Automática (Zero-Shot)
# ================================

# Lista de temas possíveis para o modelo e suas definições
# É preciso tomar cuidado com os nomes e definições. O modelo zero-shot é literal, se uma palavra-chave estiver no nome do tema ele vai associar com aquilo sem considerar o contexto
# Analise as palavras chaves e o tema que o modelo atribuiu e faça as alterações necessárias
candidatos_temas = {
    "Dramas Familiares e Íntimos":
        "Filmes sobre relacionamentos amorosos, amantes, segredos, divórcio, ou a relação entre pais, mães e filhos. Histórias de luto, amadurecimento, retorno ao lar e reencontros emocionais.",

    "Vida Urbana e Metrópole":
        "Histórias ambientadas em grandes cidades, focadas na rotina de trabalho, trânsito, violência urbana, especulação imobiliária ou na vida em apartamentos e periferias.", 
    
    "Interior, Sertão e Natureza":
        "Histórias onde o protagonista é a terra, a seca, a floresta ou a luta pela sobrevivência no campo. Foco na geografia, no isolamento, lendas rurais e comunidades, muito além de apenas um drama familiar.", 
    
    "Questões Sociais e Políticas":
        "Foco em movimentos sociais, ditadura, greves, sistema de saúde, educação, criminalidade sistêmica ou denúncias de injustiça social.", 
    
    "Investigação, Crime e Mistério":
        "Filmes de suspense, policiais, crimes reais ou a busca obsessiva por uma resposta. Envolve detetives, jornalistas ou personagens investigando um desaparecimento ou morte."
}

# Carregar o classificador (Zero-Shot)
print("Carregando modelo de classificação...")
classifier = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

# Função que usa o NMF + Zero-Shot para dar nome
def nomear_clusters(modelo_nmf, vetorizador, classificador, candidatos, threshold=0.30):
    feature_names = vetorizador.get_feature_names_out()
    mapa_nomes = {}
    
    print("\n=== IDENTIFICANDO TEMAS ===")
    
    for topic_idx, topic in enumerate(modelo_nmf.components_):
        # Pega as 10 palavras mais fortes desse cluster
        top_indices = topic.argsort()[:-11:-1]
        palavras_chave = [feature_names[i] for i in top_indices]
        texto_cluster = ", ".join(palavras_chave)
        
        # O modelo Zero-Shot lê as palavras e escolhe o melhor nome
        # Passamos list(candidatos.keys()) para garantir que ele veja os Títulos
        resultado = classificador(texto_cluster, list(candidatos.keys()), multi_label=True)
        
        # Score é o quanto o modelo tem certeza que o grupo pertence a determinado tema
        score = dict(zip(resultado['labels'], resultado['scores']))

        temas_validos = {
            tema: valor
            for tema, valor in score.items()
            if valor >= threshold
        }


        if not temas_validos:
            # O primeiro da lista de labels é o vencedor (maior score)
            melhor_tema = resultado['labels'][0]
            melhor_score = resultado['scores'][0]
            temas_validos = {melhor_tema: melhor_score}
        
        print(f"Grupo {topic_idx}:")
        print(f"   Palavras-chave: {texto_cluster}")

        for tema, valor in list(temas_validos.items())[:2]: 
            print(f"   → {tema} ({valor:.1%})")
        print()

        mapa_nomes[topic_idx] = resultado['labels'][0] 

    return mapa_nomes

# ========================================
# Executar a função e salvar no dataframe
# ========================================

dicionario_temas = nomear_clusters(nmf_model, tfidf_vectorizer, classifier, candidatos_temas)


topicos_filmes = nmf_model.transform(tfidf_matrix)
dataset_synopsis_treated['cluster_id'] = topicos_filmes.argmax(axis=1)

dataset_synopsis_treated['tema_principal'] = dataset_synopsis_treated['cluster_id'].map(dicionario_temas)

print("=== AMOSTRA DO RESULTADO ===")
print(dataset_synopsis_treated[['title', 'tema_principal']].head(10))

c:\Users\usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\decomposition\_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


Carregando modelo de classificação...


Device set to use cpu



=== IDENTIFICANDO TEMAS ===
Grupo 0:
   Palavras-chave: dois, após, fim, casa, encontro, nunca, amantes, dois amantes, anos, segredo
   → Dramas Familiares e Íntimos (74.2%)
   → Investigação, Crime e Mistério (70.7%)

Grupo 1:
   Palavras-chave: lidar, precisa lidar, precisa, porto, vera, religiosa, negra, jogador, colégio, lidar consequências
   → Vida Urbana e Metrópole (90.0%)
   → Questões Sociais e Políticas (54.0%)

Grupo 2:
   Palavras-chave: homem, tem, interior, dias, grande, isolado, solidão, homem idoso, descobre, família
   → Interior, Sertão e Natureza (67.7%)
   → Questões Sociais e Políticas (63.6%)

Grupo 3:
   Palavras-chave: comunidade, dona, dia, moradores, costumes, tradições, vez, documentário explora, única, populares
   → Vida Urbana e Metrópole (56.8%)
   → Questões Sociais e Políticas (35.6%)

Grupo 4:
   Palavras-chave: paulo, são paulo, são, gabriel, periferia, periferia são, brasileiro, difícil, erika, anos
   → Questões Sociais e Políticas (45.9%)

Grupo 

In [ ]:
dataset_master = pd.read_parquet('../data/dataset_master_treated.parquet')

dataset_master['is_winner'] = dataset_master['award_category'].apply(lambda x: 1 if len(x) > 0 else 0)

cols_to_keep = [
    'balanced_rating',
    'vote_count',
    'type',
    'genre',
    'is_winner',
    'id_movie'
]

master_selected = dataset_master[cols_to_keep]


df_master_synopsis = master_selected.merge(
    dataset_synopsis_treated[['id_movie', 'tema_principal']],
    on = 'id_movie',
    how = 'left'
)



print(df_master_synopsis.head())

   balanced_rating  vote_count type                     genre  is_winner  \
0         3.443664          24   cm           [Drama, Ficção]          1   
1         3.505517          32   cm         [Crime, Mistério]          1   
2         3.497371         330   cm          [Comédia, Drama]          1   
3         3.679374          56   cm          [Drama, Família]          1   
4         3.499560         245   cm  [Comédia, Drama, Música]          1   

                                   id_movie                tema_principal  
0                               ocorpo_2015   Dramas Familiares e Íntimos  
1                        otetosobrenos_2015   Interior, Sertão e Natureza  
2  quandopareidemepreocuparcomcanalhas_2015   Interior, Sertão e Natureza  
3                                   ba_2015       Vida Urbana e Metrópole  
4                    dalicencadecontar_2015  Questões Sociais e Políticas  


In [10]:
locking_in = df_master_synopsis[df_master_synopsis['id_movie'] == 'ausencia_2014']

print(locking_in)

    balanced_rating  vote_count type    genre  is_winner       id_movie  \
82         3.266761         444   lm  [Drama]          1  ausencia_2014   

                 tema_principal  
82  Interior, Sertão e Natureza  
